# Lake Victoria Fish Stock & Export Risk Model

This notebook imports the OOP models from `src/fishstock.py` and the reproducible analysis workflow from `src/fishstock_analysis.py`. Numerical inputs and price behavior are illustrative assignment/model assumptions, not measured Lake Victoria data.

## Fibonacci baseline

The first 15 Fibonacci numbers are shown as a historical baseline only. Successive Fibonacci ratios converge to the golden ratio [1]. Applying that accelerating sequence directly to fish abundance is biologically unrealistic: it has no carrying capacity or density dependence and omits food limits, recruitment variation, and mortality. Real fishery management uses stock and ecosystem evidence; the FAO guidebook describes MSY as a model-based reference point and emphasizes uncertainty in stock dynamics [2].

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.fishstock import FishStock, PriceModel, RiskAssessor
from src.fishstock_analysis import (
    REFERENCE_HARVEST_RATE,
    run_analysis,
)

In [ ]:
results = run_analysis(paths=1000, seed=2026)
print('Fibonacci baseline, 15 terms:', results['fibonacci_baseline'].tolist())
print(f'MSY: {results["maximum_sustainable_yield_tonnes_per_week"]:,.0f} tonnes/week')
print(f'Reference harvest rate: {REFERENCE_HARVEST_RATE:.2f}/week')

## Model and risk definitions

The discrete model updates stock as $N_{t+1}=N_t+rN_t(1-N_t/K)-hN_t$, using $r=0.4$, $K=10{,}000$ tonnes, and $N_0=4{,}000$ tonnes. With proportional harvesting, equilibrium stock is $K(1-h/r)$ when $h<r$. The theoretical maximum sustainable yield is $rK/4=1{,}000$ tonnes per week, attained at $h=r/2=0.20$ under this assignment's logistic model. FAO describes MSY as a model-based reference point and cautions that real stock productivity varies and MSY is uncertain [2].

Revenue converts harvest tonnes to kilograms before multiplying by the simulated UGX/kg price. The 5% revenue VaR is defined here as the fifth percentile of annual revenue: only 5% of simulated outcomes are lower. The report also gives expected revenue minus that percentile as an equivalent lower-tail shortfall.

Risk bands are provisional policy cutoffs: Low below 10% CV, Moderate from 10% to below 25%, and High at or above 25%. CV is dimensionless; a raw revenue variance threshold such as 50,000 is not portable because variance is measured in UGX$^2$ and changes with currency scale, harvest volume, and horizon.

In [ ]:
scenario_rows = []
for harvest_rate, scenario in results['scenarios'].items():
    risk = scenario['risk']
    scenario_rows.append({
        'Harvest rate h': harvest_rate,
        'Final stock (tonnes)': scenario['final_stock_tonnes'],
        'Total harvest (tonnes)': scenario['total_harvest_tonnes'],
        'Mean catch (tonnes/week)': scenario['mean_harvest_tonnes_per_week'],
        'MSY (tonnes/week)': results['maximum_sustainable_yield_tonnes_per_week'],
        'Mean annual revenue (UGX)': risk['mean'],
        '5% revenue VaR (UGX)': risk['var_5_revenue'],
        'Revenue CV': risk['coefficient_of_variation'],
        'Risk class': risk['risk_class'],
    })

scenario_table = pd.DataFrame(scenario_rows).set_index('Harvest rate h')
scenario_table

In [ ]:
reference_risk = results['scenarios'][REFERENCE_HARVEST_RATE]['risk']
pd.DataFrame(
    {key: reference_risk[key] for key in (
        'mean', 'median', 'variance', 'standard_deviation',
        'coefficient_of_variation', 'risk_class',
        'var_5_revenue', 'loss_var_95_vs_mean',
    )},
    index=['h=0.20'],
)

In [ ]:
closure = results['closed_season']
closed_season_table = pd.DataFrame([{
    'Harvest rate h': closure['harvest_rate'],
    'Closure weeks/year': closure['closed_weeks_per_year'],
    'Open-season final stock (tonnes)': closure['open_final_stock_tonnes'],
    'Closed-season final stock (tonnes)': closure['closed_final_stock_tonnes'],
    'Open 5-year revenue (UGX)': closure['open_mean_revenue_UGX'],
    'Closed 5-year revenue (UGX)': closure['closed_mean_revenue_UGX'],
    'Revenue change (%)': closure['mean_revenue_change_percent'],
}])
closed_season_table

## Interpretation and limitations

The scenario with the largest modeled catch need not be the safest choice: harvest near MSY depends on the assumed growth rate and carrying capacity being accurate. A seasonal closure sacrifices fishing weeks but allows stock recovery; this model compares five-year revenue under the same seeded price paths. The resulting change is model-based, not a causal forecast for Lake Victoria.

The population equation omits age structure, recruitment variability, species interactions, illegal catch, and climate effects. Harvest rate is a fixed weekly fraction outside the closed season, and prices follow a bounded zero-drift random walk; neither assumption is fitted to cooperative or market observations. The CV thresholds are transparent examples, not universal risk standards. Calibrate parameters and risk limits against local catch records, ecological surveys, realized prices, and the cooperative's ability to absorb revenue shortfalls before making policy decisions.

In [ ]:
fish_figure, fish_axis = plt.subplots(figsize=(14, 6))
fish_axis.imshow(plt.imread(results['plot_path']))
fish_axis.axis('off')
fish_figure

## References

1. Weisstein, E. W. “Fibonacci Number.” *Wolfram MathWorld*. https://mathworld.wolfram.com/FibonacciNumber.html. (Successive Fibonacci ratios converge to the golden ratio.)
2. Cochrane, K. L., ed. (2002). *A Fishery Manager’s Guidebook: Management Measures and Their Application*. FAO Fisheries Technical Paper 424, Chapter 5, especially §3.2 and §4.1. https://www.fao.org/4/y3427e/y3427e07.htm. (Surplus-production/MSY context, biological variability, and assessment limitations.)

Growth, harvest-rate, and bounded price-process values in this notebook are exercise assumptions; the FAO reference provides management context, not local Lake Victoria calibration.